<a href="https://colab.research.google.com/github/Rishitha110506/Machine-Learning/blob/main/ML_Lab_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MACHINE LEARNING ASSIGNMENT 09

NAME: KAMPALLI RISHITHA


REG NO : BL.SC.U4AIE24020

SEC : D.Sec

loading dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving BERT_Embeddings.xlsx to BERT_Embeddings.xlsx


A1  
In this code, the dataset is loaded and only numerical features are selected. The data is split into training and testing sets. Then, preprocessing is applied using scaling and PCA to normalize and reduce dimensions. A stacking classifier is built using multiple models as base learners and Logistic Regression as the meta-model. The model is trained on the training data and tested on unseen data, and accuracy is calculated. This approach improves performance by combining the strengths of multiple models.

Stacking Classifier :

A Stacking Classifier is an ensemble method where multiple models (like KNN, SVM, Random Forest, etc.) are trained together, and their predictions are combined using a final model (meta-model) to give better results.

We use stacking to improve accuracy and performance by combining different models, since each model learns different patterns. This makes the final prediction more reliable.

In [ ]:
# importing required models
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from xgboost import XGBClassifier

df = pd.read_excel("BERT_Embeddings.xlsx")

# preparing features

X = df.select_dtypes(include=['number'])   # removes text
y = df["label"]

# Fix column names issue
X.columns = X.columns.astype(str)

# splitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# pre processing

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

pca = PCA(n_components=100)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

# model stacking

base_models = [
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('svm', SVC(probability=True)),
    ('dt', DecisionTreeClassifier()),
    ('rf', RandomForestClassifier(n_estimators=50)),
    ('nb', GaussianNB()),
    ('ada', AdaBoostClassifier()),
    ('xgb', XGBClassifier(n_estimators=50, use_label_encoder=False, eval_metric='logloss')),
    ('mlp', MLPClassifier(max_iter=200))
]

meta_model = LogisticRegression()

model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    n_jobs=-1
)

# training model
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(" Stacking Accuracy:", accuracy)

 Stacking Accuracy: 0.5897435897435898


IMPROVED :   
The improved code was used to increase accuracy by selecting strong models, tuning their parameters, increasing PCA components to retain more information, and using stratified splitting. Weak models were removed to reduce noise, leading to better and more reliable predictions.

The dataset is loaded and numeric features are selected. Data is split using stratified sampling. Scaling and PCA are applied for preprocessing. A stacking classifier is built using tuned models (KNN, SVM, Random Forest, XGBoost, MLP) with Logistic Regression as meta-model. The model is trained and tested, and accuracy is calculated.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from xgboost import XGBClassifier


df = pd.read_excel("BERT_Embeddings.xlsx")


X = df.select_dtypes(include=['number'])
y = df["label"]
X.columns = X.columns.astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


pca = PCA(n_components=200)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)


base_models = [
    ('knn', KNeighborsClassifier(n_neighbors=9, weights='distance')),

    ('svm', SVC(
        C=2.0,
        kernel='rbf',
        gamma='scale',
        probability=True
    )),

    ('rf', RandomForestClassifier(
        n_estimators=150,
        max_depth=None,
        random_state=42
    )),

    ('xgb', XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss'
    )),

    ('mlp', MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=400,
        learning_rate_init=0.001
    ))
]


meta_model = LogisticRegression(max_iter=1000)

model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    n_jobs=-1
)



model.fit(X_train, y_train)



y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("High Accuracy Stacking:", accuracy)

High Accuracy Stacking: 0.971342383107089


A2

without stacking


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_excel("BERT_Embeddings.xlsx")

# Prepare features
X = df.select_dtypes(include=['number'])
y = df["label"]
X.columns = X.columns.astype(str)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=200)),
    ('classifier', RandomForestClassifier(n_estimators=100))
])

# Train
pipeline.fit(X_train, y_train)

# Test
y_pred = pipeline.predict(X_test)

# Result
print("Pipeline Accuracy:", accuracy_score(y_test, y_pred))

Pipeline Accuracy: 0.6138763197586727


with stacking

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# Load dataset
df = pd.read_excel("BERT_Embeddings.xlsx")

# Prepare features
X = df.select_dtypes(include=['number'])
y = df["label"]
X.columns = X.columns.astype(str)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Base models
base_models = [
    ('knn', KNeighborsClassifier(n_neighbors=9, weights='distance')),
    ('svm', SVC(C=2.0, probability=True)),
    ('rf', RandomForestClassifier(n_estimators=100)),
    ('xgb', XGBClassifier(n_estimators=100, eval_metric='logloss')),
    ('mlp', MLPClassifier(max_iter=300))
]

# Meta model
meta_model = LogisticRegression(max_iter=1000)

stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    n_jobs=-1
)

# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=200)),
    ('stacking', stacking_model)
])

# Train
pipeline.fit(X_train, y_train)

# Test
y_pred = pipeline.predict(X_test)

# Result
print("Pipeline + Stacking Accuracy:", accuracy_score(y_test, y_pred))

Pipeline + Stacking Accuracy: 0.9653092006033183


In [ ]:
!pip install lime


LIME (Local Interpretable Model-agnostic Explanations) is a technique used in machine learning to explain the predictions of complex models. It works by approximating the model locally (around a single prediction) using a simple, interpretable model, helping us understand which features influenced that specific decision.

LIME is used to make “black-box” models (like neural networks or ensemble models) more understandable. It helps in debugging models, increasing trust, and explaining predictions to users by showing which input features contributed the most to a particular output.

A3  
This code uses LIME to interpret a machine learning model’s prediction. First, the training and test datasets are converted into NumPy arrays because LIME requires data in numerical array format. Then, a LimeTabularExplainer is created using the training data, where feature names are assigned (as indices), class labels are defined, and the mode is set to classification. After that, a single sample from the test set is selected. The explain_instance function is then used to generate an explanation for that specific sample by calling the model’s probability prediction function (predict_proba). Finally, exp.as_list() prints the contribution of each feature, showing how they influenced the model’s prediction for that particular instance.


LIME provides local explanations, meaning it explains one prediction at a time, not the whole model. Each data point can have different feature importance, so we pick a single sample to understand **why the model made that specific decision**.


In [ ]:
import numpy as np
from lime.lime_tabular import LimeTabularExplainer

# converting data
X_train_np = np.array(X_train)
X_test_np = np.array(X_test)

# creating explainer
explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=[str(i) for i in range(X_train_np.shape[1])],
    class_names=['1', '2', '3'],
    mode='classification'
)

# selecting sample
sample = X_test_np[0]

# explanation
exp = explainer.explain_instance(
    data_row=sample,
    predict_fn=pipeline.predict_proba
)


print(exp.as_list())

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[('2.00 < 768 <= 3.00', -0.35310623690110743), ('-0.13 < 235 <= -0.01', -0.03970744653020152), ('364 <= -0.13', 0.03249490152734248), ('738 <= -0.01', -0.03137172913887789), ('-0.01 < 535 <= 0.09', 0.028791152718989216), ('677 <= 0.06', 0.02622812070789727), ('752 <= 0.11', -0.025607648745851627), ('9 <= -0.11', -0.02496401459843323), ('0.04 < 388 <= 0.14', 0.02415390054078201), ('0.01 < 483 <= 0.15', -0.021792933443447828)]


The output list shows the LIME explanation for one sample. Each tuple contains a feature range and its weight. A positive value means the feature supports the prediction, while a negative value means it opposes it. Larger values indicate stronger influence. Since the features are BERT embedding dimensions, they represent numerical patterns rather than directly understandable words.